# 01 · Document Ingestion

This notebook covers the first stage of the RAG pipeline:
1. Load PDF documents from a local directory
2. Split them into overlapping chunks using `RecursiveCharacterTextSplitter`
3. Inspect chunk statistics (size distribution, source breakdown)

The actual embedding and vector storage is handled in **02_rag_query_pipeline.ipynb**.

**Credentials**: All secrets are read from environment variables — never hard-code them.

In [ ]:
import os
from pathlib import Path

# Point this at your local PDF directory
PDF_DIR = Path(os.getenv("PDF_DIR", "../data/docs"))
print(f"PDF directory: {PDF_DIR.resolve()}")
print(f"Exists: {PDF_DIR.exists()}")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

all_chunks = []
per_file_counts = {}

for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    chunks = splitter.split_documents(pages)
    for chunk in chunks:
        chunk.metadata.setdefault("source", pdf_path.name)
    all_chunks.extend(chunks)
    per_file_counts[pdf_path.name] = len(chunks)
    print(f"  {pdf_path.name}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

## Chunk Statistics

Understanding chunk size distribution helps tune `chunk_size` and `chunk_overlap`.
Chunks that are too short lose context; chunks that are too long dilute the relevant signal.

In [ ]:
import statistics

lengths = [len(c.page_content) for c in all_chunks]

print(f"Chunk count : {len(lengths)}")
print(f"Mean length : {statistics.mean(lengths):.0f} chars")
print(f"Median      : {statistics.median(lengths):.0f} chars")
print(f"Std dev     : {statistics.stdev(lengths):.0f} chars")
print(f"Min / Max   : {min(lengths)} / {max(lengths)} chars")

In [ ]:
# Top 5 largest chunks — inspect for runaway pages
top5 = sorted(all_chunks, key=lambda c: len(c.page_content), reverse=True)[:5]
for i, chunk in enumerate(top5):
    print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) from {chunk.metadata.get('source')} ---")
    print(chunk.page_content[:300], "...")

## Source Breakdown

Confirm each file contributed a reasonable share of chunks.

In [ ]:
from collections import Counter

source_counts = Counter(c.metadata["source"] for c in all_chunks)
for source, count in source_counts.most_common():
    bar = "█" * (count // 2)
    print(f"{source:<50} {count:>4}  {bar}")

## Next Step

The `all_chunks` list is ready for embedding. Open **02_rag_query_pipeline.ipynb** to:
- Authenticate with OCI GenAI
- Embed chunks using Cohere `embed-english-v3.0`
- Store vectors in Oracle Autonomous Database 23ai